## Exploración inicial de datos

### By:
Leydy Osorio Vargas

### Date:
2026-08-28

### Description:

Este notebook realiza la exploración estructural inicial del conjunto de datos de enfermedad cardiaca almacenado en formato RAW. El objetivo es comprender su esquema, identificar las formas utilizadas para representar valores nulos, validar la naturaleza de cada variable y convertir las columnas a tipos de datos correctos y uniformes.

Como resultado, se generará un dataset intermedio en formato Parquet que servirá como entrada para el análisis exploratorio de datos de la siguiente etapa. En esta actividad no se realizarán imputaciones, transformaciones de atributos ni modelamiento.

## 📚 Import  libraries

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa


## 💾 Load data

In [2]:
PROJECT_DIR = Path.cwd().resolve()

if not (PROJECT_DIR / "data").exists():
    PROJECT_DIR = PROJECT_DIR.parents[1]

DATA_DIR = PROJECT_DIR / "data"
RAW_FILE = DATA_DIR / "01_raw" / "corazon.csv"

heart_df = pd.read_csv(RAW_FILE, low_memory=False)

print(f"Archivo cargado: {RAW_FILE}")
print(f"Dimensiones: {heart_df.shape[0]} filas y {heart_df.shape[1]} columnas")


Archivo cargado: /mnt/c/Users/Usuario/Heart-Project/data/01_raw/corazon.csv
Dimensiones: 3030 filas y 14 columnas


In [3]:
heart_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 3030 entries, 0 to 3029
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   age         3000 non-null   str    
 1   sex         2969 non-null   str    
 2   chest_pain  2947 non-null   str    
 3   rest_bp     2949 non-null   str    
 4   chol        2945 non-null   str    
 5   fbs         2933 non-null   float64
 6   rest_ecg    2837 non-null   str    
 7   max_hr      2859 non-null   str    
 8   exang       2879 non-null   str    
 9   old_peak    2880 non-null   str    
 10  slope       2879 non-null   str    
 11  ca          2868 non-null   str    
 12  thal        2904 non-null   str    
 13  disease     2924 non-null   str    
dtypes: float64(1), str(13)
memory usage: 506.9 KB


In [4]:
heart_df.sample(10, random_state=42)


,age,sex,chest_pain,rest_bp,chol,fbs,rest_ecg,max_hr,exang,old_peak,slope,ca,thal,disease
1207,45,Male,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,reversable,1
256,67,Female,asymptomatic,106,223,0.0,normal,142,0,0.3,1,2.0,normal,0
2356,54,Male,asymptomatic,122,286,0.0,left ventricular hypertrophy,116,1,3.2,2,2.0,normal,1
175,57,Male,asymptomatic,152,274,0.0,normal,88,1,1.2,2,1.0,reversable,1
211,38,Male,typical,120,231,0.0,normal,182,1,3.8,2,0.0,reversable,1
411,61,Male,asymptomatic,120,260,0.0,normal,140,1,3.6,2,1.0,reversable,1
52,44,Male,asymptomatic,112,290,0.0,left ventricular hypertrophy,153,0,0.0,1,1.0,normal,1
266,52,Male,asymptomatic,128,204,1.0,normal,156,1,1.0,2,0.0,NaN,1
479,52,Male,asymptomatic,108,233,1.0,normal,147,0,0.1,1,3.0,reversable,0
1292,45,NaN,NaN,NaN,NaN,NaN,NaN,148,1,3.0,2,0.0,normal,0


### Descripción inicial

El conjunto de datos contiene 3.030 registros y 14 variables relacionadas con características demográficas, síntomas, resultados de exámenes y presencia de enfermedad cardiaca.

La inspección inicial evidencia valores faltantes en todas las variables. Además, la mayoría de las columnas fueron interpretadas como texto, incluso aquellas que conceptualmente deberían ser numéricas. Por lo tanto, será necesario revisar la representación de los valores nulos y validar los tipos antes de realizar el análisis exploratorio.

## 🔍 Identificación y unificación de valores nulos

In [5]:
missing_summary = pd.DataFrame(
    {
        "data_type": heart_df.dtypes.astype(str),
        "missing_values": heart_df.isna().sum(),
        "missing_percentage": (heart_df.isna().mean() * 100).round(2),
    }
).sort_values("missing_percentage", ascending=False)

missing_summary


,data_type,missing_values,missing_percentage
rest_ecg,str,193,6.37
max_hr,str,171,5.64
ca,str,162,5.35
exang,str,151,4.98
slope,str,151,4.98
old_peak,str,150,4.95
thal,str,126,4.16
disease,str,106,3.50
fbs,float64,97,3.20
chol,str,85,2.81


Todas las variables presentan valores faltantes. La mayor proporción se encuentra en `rest_ecg` con 6,37 %, mientras que `age` presenta la menor proporción con 0,99 %. Ninguna columna supera el 7 % de datos faltantes.

En esta etapa los valores faltantes no serán imputados ni se eliminarán registros. Únicamente se verificará cómo están representados en el archivo RAW y se unificará su representación para facilitar las etapas posteriores.

In [6]:
raw_text_df = pd.read_csv(
    RAW_FILE,
    dtype="string",
    keep_default_na=False,
)

trimmed_df = raw_text_df.apply(lambda column: column.str.strip())

possible_null_markers = [
    "",
    "?",
    "NA",
    "N/A",
    "NULL",
    "null",
    "None",
    "none",
    "NaN",
    "nan",
]

null_marker_summary = pd.DataFrame(
    {
        marker if marker else "<empty>": trimmed_df.eq(marker).sum()
        for marker in possible_null_markers
    }
)

null_marker_summary = null_marker_summary.loc[
    :,
    null_marker_summary.sum() > 0,
]

null_marker_summary


,<empty>
age,30
sex,61
chest_pain,83
rest_bp,81
chol,85
fbs,97
rest_ecg,193
max_hr,171
exang,151
old_peak,150


Al cargar el archivo RAW sin la detección automática de valores nulos, se encontró que todos los datos faltantes están representados mediante campos vacíos. No se identificaron otros marcadores como `?`, `NA`, `N/A`, `NULL` o `None`.

Para garantizar una representación uniforme, los campos vacíos se reemplazarán explícitamente por `pd.NA`. También se eliminarán espacios accidentales al inicio o al final de los valores de texto.

In [7]:
heart_df = trimmed_df.replace("", pd.NA)

heart_df.isna().sum()


age            30
sex            61
chest_pain     83
rest_bp        81
chol           85
fbs            97
rest_ecg      193
max_hr        171
exang         151
old_peak      150
slope         151
ca            162
thal          126
disease       106
dtype: int64

## 🔧 Validación y corrección de tipos de datos

## 📊 Analysis of Results and Conclusions 

Description of the results obtained and if there are conclusions that can be drawn from them.

The analysis of results must be related to the description of the task.

**Note:** An analysis of results does not necessarily lead to conclusions, but to ideas or proposals for future work


## 💡 Proposals and Ideas

From the results obtained, what ideas or proposals can be generated to continue with the project


## 📖 References